# <font color='264CC7'> activity_tracker </font>

Este notebook tiene como objetivo desarrollar y validar progresivamente los módulos que conforman el sistema telegram-notion-activity-tracker. Cada sección corresponde a un componente funcional independiente, que será probado de manera aislada antes de integrarse en el flujo completo.

El sistema estará compuesto por los siguientes módulos:
- Clasificador: analiza el texto de la actividad y lo categoriza según la taxonomía definida.
- Transcriptor: convierte mensajes de voz en texto procesable.
- Envío a Notion: construye el registro estructurado y lo inserta en la base de datos.
- Bot de Telegram: recibe mensajes y audios, y activa el flujo de procesamiento.
- Pipeline: integra todos los módulos en un proceso completo desde la captura hasta el almacenamiento.

Los paquetes necesarios son:

In [1]:
# !pip install pymupdf 
# !pip install openai
# !pip install notion-client
# !pip install pyTelegramBotAPI

---
## <font color='264CC7'> Clasificador </font>


In [1]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field
import json

In [2]:
# Configurar la clave de API
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
def load_categories(file_path: str) -> dict:
    with open(file_path, 'r', encoding='utf-8') as file:
        categories = json.load(file)
    return categories

categories = load_categories('categories.json')
print(categories)

{'Carreras-Programas': {'nombre': 'Diseño, seguimiento y evaluación de carreras y programas', 'descripcion': 'Actividades relacionadas con sílabos, carreras, programas, maestrías o contratación docente general vinculada al funcionamiento académico global de la Facultad.', 'usar_cuando': 'Usar cuando la actividad trate sobre revisión de sílabos, planificación de carreras o programas, reuniones de maestrías o procesos de contratación docente a nivel general.'}, 'Internacionalizacion': {'nombre': 'Internacionalización curricular y movilidad docente', 'descripcion': 'Acciones relacionadas explícitamente con internacionalización curricular, cooperación internacional o movilidad docente.', 'usar_cuando': 'Usar cuando la actividad mencione claramente movilidad docente, cooperación internacional o internacionalización del currículo.'}, 'Recursos': {'nombre': 'Gestión de recursos académicos y logísticos', 'descripcion': 'Actividades relacionadas con presupuesto, logística, traspaso de informaci

In [4]:
class Activity(BaseModel):
    name: str
    description: str
    date: str = Field(description="Fecha en formato AAAA-MM-DD.")
    category: str

class ActivityBatch(BaseModel):
    activities: list[Activity] = Field(description="Lista de actividades detectadas. Puede contener una o más actividades.")

def classify_activities(texto: str, categories: dict, today: str) -> list[Activity]:
    allowed = list(categories.keys())

    category_block = "\n".join(
        [
            (
                f"- {key}:\n"
                f"  nombre: {value.get('nombre', '')}\n"
                f"  descripcion: {value.get('descripcion', '')}\n"
                f"  usar_cuando: {value.get('usar_cuando', '')}"
            )
            for key, value in categories.items()
        ]
    )
    system_prompt = (
        "Eres un asistente para registrar actividades de gestión académica de la USDIC en la FCENA (PUCE). "
        "Tu tarea es detectar actividades concretas dentro de un texto, separarlas si hay más de una, "
        "clasificarlas en exactamente una categoría permitida y devolver la salida ajustada al esquema indicado.\n\n"
        "Contexto institucional:\n"
        "- FCENA incluye las carreras de Biología, Microbiología, Química, Bioingeniería, Ciencias Biomédicas y Ciencia de Datos.\n"
        "- También incluye programas de posgrado como Maestría en Biología y Maestría en Ciencias Actuariales.\n"
        "- Rutas académicas: procesos para que un estudiante apruebe un resultado de aprendizaje (RdA) pendiente.\n"
        "- Banner: sistema de gestión académica utilizado para matrícula, registro de notas y programación académica.\n"
        "- EVA: Entorno Virtual de Aprendizaje (aulas virtuales).\n\n"
        "Reglas generales:\n"
        "- No inventes información.\n"
        "- Si algo no está explícito, mantén la redacción general.\n"
        "- Cada actividad debe representar una acción concreta y diferenciable.\n"
        "- Cada actividad debe tener exactamente una categoría.\n"
        "- Clasifica según el objetivo principal de la actividad, no por palabras aisladas.\n"
        "- Si el texto menciona reunión, apoyo, gestión, revisión o coordinación, clasifica por el propósito principal de esa acción.\n"
        "- Usa la categoría 'Otras' solo como último recurso.\n"
    )
    user_prompt = (
        "Analiza el texto y devuelve solo el objeto del esquema con 'activities' como lista. "
        "Si hay una sola actividad, igualmente devuélvela dentro de la lista.\n\n"
        "Instrucciones:\n"
        "1) Detecta si el texto contiene una o varias actividades.\n"
        "2) Separa actividades solo si describen acciones realmente distintas.\n"
        "3) Para cada actividad, elige EXACTAMENTE una categoría de la lista permitida.\n"
        "4) Para cada actividad, genera un nombre corto, simple y administrativo de 3 a 8 palabras.\n"
        "5) Para cada actividad, redacta una descripción breve de una sola frase.\n"
        f"6) Si no hay fecha explícita, usa la de hoy: {today} (formato AAAA-MM-DD).\n"
        "7) Si hay una fecha explícita en el texto, conviértela a formato AAAA-MM-DD.\n"
        "8) No inventes responsables, detalles ni fechas adicionales.\n"
        "9) Prioriza la mejor categoría posible antes de usar 'Otras'.\n\n"
        f"Categorías permitidas: {allowed}\n\n"
        "Definición operativa de categorías:\n"
        f"{category_block}\n\n"
        "Reglas de desambiguación:\n"
        "- Si trata sobre sílabos, carreras, programas, maestrías o contratación docente general, prioriza 'Carreras-Programas'.\n"
        "- Si trata sobre presupuesto, logística, traspaso de información, metodologías organizativas o espacios físicos, prioriza 'Recursos'.\n"
        "- Si trata sobre rutas académicas, calificaciones, notas, recuperación académica o aprobación de RdA, prioriza 'RdA'.\n"
        "- Si trata sobre coordinación de asignaturas, componentes prácticos o articulación entre docentes de una materia, prioriza 'Áreas'.\n"
        "- Si trata sobre talleres, charlas o actividades formativas para docentes, prioriza 'Capacitacion'.\n"
        "- Si trata sobre seminario de titulación, integración curricular o prácticas preprofesionales, prioriza 'Titulacion-Practicas'.\n"
        "- Si trata sobre horarios, asignación docente, distribución de estudiantes, matrícula, inscripción o ajustes de oferta académica, prioriza 'Programacion'.\n"
        "- Usa 'Internacionalizacion', 'Evaluacion docente', 'Calidad' y 'Aulas virtuales' solo cuando el texto lo indique de forma explícita.\n\n"
        f"Texto a analizar:\n{texto}"
    )
    response = client.responses.parse(
        model="gpt-4.1-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        text_format=ActivityBatch,
        temperature=0,
    )
    parsed = response.output_parsed
    if not parsed or not parsed.activities:
        raise ValueError("No se detectaron actividades válidas en la respuesta del modelo.")
    return parsed.activities

In [ ]:
# texto = "Ayer generé los archivos de horarios de las diferentes carreras de la facultad que fueron enviado a estudiantes y a la DDE."

# actividad_clasificada = classify_activities(texto, categories, "2026-20-02")
# print(actividad_clasificada)

[Activity(name='Generación y envío de horarios', description='Se generaron los archivos de horarios de las diferentes carreras y se enviaron a estudiantes y a la DDE.', date='2026-02-20', category='Programacion')]


In [7]:
# texto = "Solicione un inconveniente reportado por estudiantes en el que no se habían pasado las notas de los itinerarios al sistema Banner"

# actividad_clasificada = classify_activity(texto, categories, "2026-20-02")
# print(actividad_clasificada)

---
## <font color='264CC7'> Transcriptor </font>

In [8]:
import os
from pathlib import Path
from openai import OpenAI

In [9]:
# Configurar la clave de API
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [10]:
def transcribe_audio(audio_path: str) -> str:
    p = Path(audio_path)
    if not p.exists():
        raise FileNotFoundError(f"No existe el archivo: {audio_path}")

    transcription = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe", 
        file=p.open("rb")
    )

    return transcription.text


In [11]:
# txt = transcribe_audio("datos/Audio.m4a")
# print("El texto transcrito es:", txt)

---
## <font color='264CC7'> Envío a Notion </font>

In [6]:
from pydantic import BaseModel, Field
import os
from notion_client import Client as NotionClient

In [7]:
NOTION_DATABASE_ID = os.getenv("NOTION_DATABASE_ID")
notion = NotionClient(auth=os.getenv("NOTION_TOKEN"))

In [8]:
class Activity(BaseModel):
    name: str
    description: str
    date: str = Field(description="Fecha en formato AAAA-MM-DD.")
    category: str

class ActivityBatch(BaseModel):
    activities: list[Activity] = Field(description="Lista de actividades detectadas. Puede contener una o más actividades.")

# Defino una Activity 
actividad = Activity(
    name='Generación y envío de horarios',
    description='Se generaron los archivos de horarios de las carreras y se enviaron a estudiantes y a la DDE.',
    date='2026-02-19',
    category='Programacion'
)
actividad_batch = ActivityBatch(activities=[actividad])

In [9]:
def enviar_a_notion(actividad: Activity):
    propiedades = {
        "Actividad": {"title": [{"text": {"content": actividad.name}}]},
        "Categoría": {"select": {"name": actividad.category}},
        "Fecha": {"date": {"start": actividad.date}},
        "Descripción": {"rich_text": [{"text": {"content": actividad.description}}]}
    }
    try:
        notion.pages.create(
            parent={"database_id": NOTION_DATABASE_ID},
            properties=propiedades
        )
    except Exception as e:
        print(f"Error al enviar a Notion: {e}")
        return {"ok": False, "error": str(e)}

    return {"ok": True}

def enviar_varias_a_notion(actividades: list[Activity]):
    resultados = []
    errores = []

    for actividad in actividades:
        resultado = enviar_a_notion(actividad)
        resultados.append(resultado)
        if not resultado.get("ok"):
            errores.append({"activity": actividad.name, "error": resultado.get("error", "Error desconocido")})

    return {
        "ok": len(errores) == 0,
        "total": len(actividades),
        "guardadas": len(actividades) - len(errores),
        "errores": errores,
    }


In [10]:
enviar_varias_a_notion(actividad_batch.activities)

{'ok': True, 'total': 1, 'guardadas': 1, 'errores': []}

---
## <font color='264CC7'> Bot de Telegram </font>

In [17]:
import telebot
from datetime import datetime

In [18]:
USUARIO_AUTORIZADO = int(os.getenv("USUARIO_AUTORIZADO"))
bot = telebot.TeleBot(os.getenv("TELEGRAM_TOKEN"))

In [ ]:
def es_usuario_autorizado(message) -> bool:
    return str(message.from_user.id) == str(USUARIO_AUTORIZADO)


def acceso_restringido(func):
    def wrapper(message):
        if not es_usuario_autorizado(message):
            bot.reply_to(message, "⛔ No estás autorizado para usar este bot.")
            return
        return func(message)
    return wrapper


def _fecha_hoy() -> str:
    return datetime.now().strftime("%Y-%m-%d")


@bot.message_handler(commands=["start"])
@acceso_restringido
def start(message):
    bot.send_message(
        message.chat.id,
        "👋 ¡Hola! Cuéntame qué actividades realizaste hoy"
    )


@bot.message_handler(content_types=["text"])
@acceso_restringido
def handle_text(message):
    texto = (message.text or "").strip()
    if not texto:
        bot.reply_to(message, "Envía una descripción de la actividad (texto).")
        return

    bot.send_message(message.chat.id, "📌 Registrando actividad...")

    try:
        activities = classify_activities(
            texto=texto,
            categories=categories,
            today=_fecha_hoy()
        )
        result = enviar_varias_a_notion(activities)
        if result.get("ok"):
            bot.send_message(message.chat.id, f"✅ {result.get('guardadas', 0)} de {result.get('total', 0)} actividades registradas en Notion.")
            for activity in activities:
                bot.send_message(
                    message.chat.id,
                    "✅ Actividad registrada en Notion: \n"
                    f"📌 Nombre: {activity.name}\n"
                    f"📌 Categoría: {activity.category}\n"
                )
        else:
            bot.send_message(message.chat.id, f"⚠️ No se pudo registrar: {result.get('error', 'Error desconocido')}")
    except Exception as e:
        bot.send_message(message.chat.id, f"⚠️ Ocurrió un error: {str(e)}")


@bot.message_handler(content_types=["voice"])
@acceso_restringido
def handle_voice(message):
    bot.send_message(message.chat.id, "🎙️ Recibido. Transcribiendo y registrando...")

    file_info = bot.get_file(message.voice.file_id)
    downloaded = bot.download_file(file_info.file_path)

    os.makedirs("temp", exist_ok=True)
    ruta_audio = os.path.join("temp", f"voice_{message.message_id}.ogg")
    with open(ruta_audio, "wb") as f:
        f.write(downloaded)

    try:
        texto = transcribe_audio(
            audio_path=ruta_audio
        )
        activities = classify_activities(
            texto=texto,
            categories=categories,
            today=_fecha_hoy()
        )
        result = enviar_varias_a_notion(activities)
        if result.get("ok"):
            bot.send_message(
                message.chat.id,
                f"✅ {result.get('guardadas', 0)} de {result.get('total', 0)} actividades registradas en Notion."
            )
            for activity in activities:
                bot.send_message(
                    message.chat.id,
                    "✅ Actividad registrada en Notion: \n"
                    f"📌 Nombre: {activity.name}\n"
                    f"📌 Categoría: {activity.category}\n"
                )
        else:
            bot.send_message(message.chat.id, f"⚠️ No se pudo registrar: {result.get('error', 'Error desconocido')}")
    except Exception as e:
        bot.send_message(message.chat.id, f"⚠️ Ocurrió un error: {str(e)}")
    finally:
        try:
            os.remove(ruta_audio)
        except OSError:
            pass


@bot.message_handler(func=lambda message: True)
def fallback(message):
    if not es_usuario_autorizado(message):
        return
    bot.reply_to(message, "Envía una actividad como texto o una nota de voz.")


# Iniciar el bot
bot.polling(none_stop=True)